# Lesson 1: Role-Based Prompting (Agent Personas)

## Historical Figure Interviewer

In this hands-on exercise, you will craft prompts to instruct an AI to convincingly adopt the persona of a historical figure during an interactive Q&A session.

## **Outline:**
- Plain Prompt: Try a basic prompt without any specific role defined
- Baseline Historical Figure Prompt: Ask the AI to role-play as a historical figure
- Define Persona-Specific Attributes: Define personality, etc.
- Add Tone and Style Specifications: Specify tone and style for the responses
- Q&A Session Format: Conduct mock Q&A sessions to test persona consistency
- Reflection & Transfer: Evaluate realism of AI responses

## API configuration

Run this notebook with the repository's Python environment. Set `OPENAI_API_KEY` for
direct OpenAI access, or set `VOCAREUM_API_KEY` for the Udacity classroom gateway.
The notebook loads the nearest local `.env` from the working directory or its parents
and preserves variables already set in the kernel environment.

When both keys are available, direct OpenAI is the default. Set
`OPENAI_BASE_URL=https://openai.vocareum.com/v1` to select Vocareum explicitly.
Only the official OpenAI and Vocareum endpoints are supported.
A classroom key stored in `OPENAI_API_KEY` is recognized by its `voc-` prefix.
Keep credentials in the environment or ignored `.env` file.

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI, OpenAIError

In [ ]:
load_dotenv(find_dotenv(usecwd=True))

openai_url = "https://api.openai.com/v1"
vocareum_url = "https://openai.vocareum.com/v1"
openai_key = (os.getenv("OPENAI_API_KEY") or "").strip()
vocareum_key = (os.getenv("VOCAREUM_API_KEY") or "").strip()
base_url = (os.getenv("OPENAI_BASE_URL") or "").strip().rstrip("/")

if base_url and base_url not in {openai_url, vocareum_url}:
    raise ValueError("OPENAI_BASE_URL must use the official OpenAI or Vocareum endpoint.")

if openai_key.startswith("voc-"):
    vocareum_key = vocareum_key or openai_key
    openai_key = ""

if base_url == vocareum_url or (not base_url and not openai_key and vocareum_key):
    api_key = vocareum_key
    base_url = vocareum_url
else:
    api_key = openai_key

if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or VOCAREUM_API_KEY for the selected provider.")

if base_url == vocareum_url and api_key.startswith("sk-"):
    raise RuntimeError("Use a Vocareum key for the classroom endpoint.")

client = OpenAI(api_key=api_key, base_url=base_url or openai_url)

In [ ]:
# Helper function to display responses as Markdown, horizontally
def display_responses(*args):
    markdown_string = "<table><tr>"
    # Headers
    for arg in args:
        markdown_string += f"<th>System Prompt:<br />{arg['system_prompt']}<br /><br />"
        markdown_string += f"User Prompt:<br />{arg['user_prompt']}</th>"
    markdown_string += "</tr>"
    # Rows
    markdown_string += "<tr>"
    for arg in args:
        markdown_string += f"<td>Response:<br />{arg['response']}</td>"
    markdown_string += "</tr></table>"
    display(Markdown(markdown_string))

In [ ]:
# Function to interact with the OpenAI API
from enum import Enum


class OpenAIModels(str, Enum):
    GPT_4O_MINI = "gpt-4o-mini"
    GPT_41_MINI = "gpt-4.1-mini"
    GPT_41_NANO = "gpt-4.1-nano"


MODEL = OpenAIModels.GPT_41_NANO


def get_completion(system_prompt, user_prompt, model=MODEL):
    """
    Function to get a completion from the OpenAI API.
    Args:
        system_prompt: The system prompt
        user_prompt: The user prompt
        model: The model to use (default is gpt-4.1-nano)
    Returns:
        The completion text
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.7,
        )
        return response.choices[0].message.content
    except OpenAIError as exc:
        raise RuntimeError(
            f"The model request failed ({type(exc).__name__}). "
            "Check your provider configuration and model access."
        ) from None


## 1. Plain Prompt

First, let's see what the model produces with a basic prompt before asking it to role-play as a historical figure.

In [ ]:
control_system_prompt = "You are a helpful assistant."
user_prompt = "Can you tell me about relativity?"

print(f"Sending prompt to {MODEL} model...")
control_response = get_completion(control_system_prompt, user_prompt)
print("Response received!\n")

display_responses(
    {
        "system_prompt": control_system_prompt,
        "user_prompt": user_prompt,
        "response": control_response,
    }
)

## 2. Baseline Historical Figure Prompt

First, let's see what the model produces with a basic prompt asking it to role-play as Albert Einstein.

In [ ]:
baseline_system_prompt = "Portray Albert Einstein in a simulated historical interview about your life and work."

print(f"Sending prompt to {MODEL} model...")
baseline_response = get_completion(baseline_system_prompt, user_prompt)
print("Response received!\n")

display_responses(
    {
        "system_prompt": baseline_system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response,
    }
)

## 3. Define Persona-Specific Attributes (3 min)

Now, let's enhance the prompt by adding specific attributes about Einstein's personality, era-specific vocabulary, and expertise.

In [ ]:
persona_system_prompt = f"""{baseline_system_prompt}

Adopt these persona characteristics:

- Personality: Curious, modest, and willing to question familiar assumptions.
- Speech style: Use plain language and concrete thought experiments.
- Expertise: Draw on special and general relativity and explain the distinction.
- Historical context: Speak as Einstein in 1950. Avoid discoveries after that date
  and acknowledge uncertainty about personal details you cannot support."""

print(f"Sending prompt to {MODEL} model...")
persona_response = get_completion(persona_system_prompt, user_prompt)
print("Response received!\n")

# Show last two prompts and responses
display_responses(
    {
        "system_prompt": baseline_system_prompt,
        "user_prompt": user_prompt,
        "response": baseline_response,
    },
    {
        "system_prompt": persona_system_prompt,
        "user_prompt": user_prompt,
        "response": persona_response,
    },
)

## 4. Add Tone and Style Specifications (3 min)

Let's further refine the prompt by adding specifications for tone and conversational style.

In [ ]:
tone_system_prompt = f"""{persona_system_prompt}

Tone and style:
- Speak warmly and conversationally to an interviewer without a physics background.
- Use one short thought experiment involving a train or an elevator.
- Explain what the example illustrates in one sentence before returning to the question."""

print("Sending prompt with tone and style specifications...")
tone_response = get_completion(tone_system_prompt, user_prompt)
print("Response received!\n")

# Display the last two prompts and responses
display_responses(
    {
        "system_prompt": persona_system_prompt,
        "user_prompt": user_prompt,
        "response": persona_response,
    },
    {
        "system_prompt": tone_system_prompt,
        "user_prompt": user_prompt,
        "response": tone_response,
    },
)

## 5. Q&A Session Format (2 min)

Now, let's incorporate a specific Q&A format to test the persona consistency.

In [ ]:
interview_prompt = """Questions:
1. What puzzle about light and motion first drew you to relativity?
2. How would you explain gravitational time dilation to a curious student?
3. What advice would you give an aspiring physicist in 1950?

Number each answer to match the questions."""

print("Sending prompt with Q&A format...")
qa_response = get_completion(tone_system_prompt, interview_prompt)
print("Response received!\n")

display_responses(
    {
        "system_prompt": tone_system_prompt,
        "user_prompt": interview_prompt,
        "response": qa_response,
    },
)

## 6. Reflection & Transfer (5 min)

Let's reflect on what we've learned from this exercise.

### Which refinement would make the persona more convincing?

The persona attributes are the strongest starting point. A date, a defined area of expertise,
and a specific way of explaining ideas give the interviewer more to compare than the figure's
name alone. The tone instructions help make the answer accessible, while the Q&A tests whether
the same voice carries across several topics.

When reviewing a live run, I would check for anachronisms, factual accuracy, and consistent
voice. A convincing simulation still needs to be checked against historical sources.
For a different historical figure, I would keep the baseline question fixed and change the
date, expertise, and style one at a time.

## Summary

In this exercise, we explored how role-based prompting can be used to create convincing historical figure personas:

1. **Plain Prompt**: We started with a simple request involving no role-playing.
2. **Baseline Prompt**: We then used a simple request for the AI to role-play as Albert Einstein.
3. **Persona-Specific Attributes**: We added details about personality traits, speech style, expertise, and historical context.
4. **Tone and Style**: We further refined the prompt with specific instructions about conversational tone and linguistic style.
5. **Q&A Format**: We tested the persona with specific questions to evaluate consistency and authenticity.

Great work on making it this far! 🎉🚀